In [1]:
# Import Required Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

import os
print("Loading Kaggle input files...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Loading Kaggle input files...
/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
# Load Data
print("\nLoading training and test data...")
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")


Loading training and test data...
Training data shape: (594194, 21)
Test data shape: (254655, 20)


In [3]:
# Preprocessing Function
def preprocess_data(df, is_training=True, encoder=None, scaler=None, target_col='Churn'):
    """
    Preprocess customer churn data with consistent encoding for train/test sets.
    """
    df_processed = df.copy()
    
    # Identify feature columns (exclude id and target)
    feature_cols = [col for col in df_processed.columns if col not in ['id', target_col]]
    
    # Separate numeric and categorical features
    numeric_cols = df_processed[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_processed[feature_cols].select_dtypes(include=['object']).columns.tolist()
    
    # Handle categorical variables with one-hot encoding
    if is_training:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
        encoder = {col: df_processed[col].unique() for col in categorical_cols}
    else:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
    
    # Combine numeric and encoded categorical features
    X = pd.concat([df_processed[numeric_cols].reset_index(drop=True), 
                   df_dummies.reset_index(drop=True)], axis=1)
    
    # Handle missing values and scaling
    if is_training:
        numeric_means = X[numeric_cols].mean()
        X[numeric_cols] = X[numeric_cols].fillna(numeric_means)
        scaler = StandardScaler()
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    else:
        if numeric_cols:
            X[numeric_cols] = scaler.transform(X[numeric_cols])
    
    if is_training:
        y = df_processed[target_col]
        return X, y, encoder, scaler
    else:
        return X, encoder, scaler

# Apply preprocessing to training data
print("\nPreprocessing training data...")
X_train, y_train, encoder, scaler = preprocess_data(train_df, is_training=True)
print(f"Training data shape: {X_train.shape}")
print(f"Target distribution:\n{y_train.value_counts()}")

# Apply preprocessing to test data
print("\nPreprocessing test data...")
X_test, _, _ = preprocess_data(test_df, is_training=False, encoder=encoder, scaler=scaler)

# Ensure test data has same columns as training data
missing_cols = set(X_train.columns) - set(X_test.columns)
extra_cols = set(X_test.columns) - set(X_train.columns)

if missing_cols:
    for col in missing_cols:
        X_test[col] = 0

if extra_cols:
    X_test = X_test.drop(columns=extra_cols)

X_test = X_test[X_train.columns]
print(f"Final test data shape: {X_test.shape}")


Preprocessing training data...
Training data shape: (594194, 30)
Target distribution:
Churn
No     460377
Yes    133817
Name: count, dtype: int64

Preprocessing test data...
Final test data shape: (254655, 30)


In [4]:
# Train Model with Optimized Hyperparameters
print("\nTraining model with optimized hyperparameters...")
print("="*70)
print("OPTIMIZED HYPERPARAMETERS")
print("="*70)

# Best hyperparameters found from tuning
best_params = {
    'learning_rate': 0.15,
    'max_depth': 7,
    'max_leaf_nodes': 50,
    'min_samples_leaf': 20,
    'l2_regularization': 0.1,
    'max_bins': 128,
    'random_state': 42
}

for param, value in best_params.items():
    print(f"  {param}: {value}")

# Train the optimized model
model = HistGradientBoostingClassifier(**best_params)
model.fit(X_train, y_train)

# Display training accuracy
train_accuracy = model.score(X_train, y_train)
print(f"\nTraining Accuracy: {train_accuracy:.6f}")
print("="*70)


Training model with optimized hyperparameters...
OPTIMIZED HYPERPARAMETERS
  learning_rate: 0.15
  max_depth: 7
  max_leaf_nodes: 50
  min_samples_leaf: 20
  l2_regularization: 0.1
  max_bins: 128
  random_state: 42

Training Accuracy: 0.862545


In [5]:
# Generate Predictions
print("\nGenerating predictions on test data...")

# Get probability predictions
predictions_proba = model.predict_proba(X_test)
churn_probability = predictions_proba[:, 1]  # Probability of churn class (Yes)

print(f"Predictions shape: {churn_probability.shape}")
print(f"Probability range: [{churn_probability.min():.6f}, {churn_probability.max():.6f}]")
print(f"Probability mean: {churn_probability.mean():.6f}")
print(f"Probability std: {churn_probability.std():.6f}")


Generating predictions on test data...
Predictions shape: (254655,)
Probability range: [0.000298, 0.987983]
Probability mean: 0.218177
Probability std: 0.274585


In [6]:
# Create and Save Submission File
print("\n" + "="*70)
print("CREATING SUBMISSION FILE")
print("="*70)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Churn': churn_probability
})

# Save submission file
submission_df.to_csv('submission.csv', index=False)
print(f"\n✓ Submission file saved: submission.csv")
print(f"  File size: {submission_df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"  Number of predictions: {len(submission_df)}")

# Display preview
print(f"\nSubmission Preview (first 15 rows):")
print(submission_df.head(15).to_string())

print("\n" + "="*70)
print("SUBMISSION READY!")
print("="*70)


CREATING SUBMISSION FILE

✓ Submission file saved: submission.csv
  File size: 3979.11 KB
  Number of predictions: 254655

Submission Preview (first 15 rows):
        id     Churn
0   594194  0.055955
1   594195  0.000807
2   594196  0.114925
3   594197  0.003847
4   594198  0.481876
5   594199  0.184274
6   594200  0.890581
7   594201  0.004012
8   594202  0.030498
9   594203  0.319051
10  594204  0.004990
11  594205  0.026109
12  594206  0.151764
13  594207  0.828703
14  594208  0.002085

SUBMISSION READY!
